# IVR: Iterative Visual Retracing
## 基于迭代视觉追溯的 VLM 幻觉缓解——自动驾驶场景感知

Kaggle 2x T4 | MiniCPM-V-4.6 + Thinking | BDD100K + COCO | 断点续跑

## 1. 环境安装

In [ ]:
!pip install -q "transformers[torch]>=5.7.0" torchvision av pyyaml rouge-score matplotlib Pillow tqdm

## 2. 拉取项目代码

从 GitHub 拉取 IVR 项目代码，无需手动上传

In [ ]:
import os

REPO_URL = "https://github.com/JKpink/yyz-project.git"
REPO_DIR = "/kaggle/working/yyz-project"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull origin main

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# 检查项目文件
!echo "=== 项目结构 ===" && ls -R {REPO_DIR}/src/ {REPO_DIR}/configs/

## 3. 加载模型

In [ ]:
# B1/B3/B4 标准模型（GPU 0）
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="cuda:0",
)
base_processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
print(f"Base model VRAM: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

## 4. 初始化 Baseline

In [ ]:
from ivr import IVRInference
from baselines.baseline_direct import BaselineDirect
from baselines.baseline_cot import BaselineThinking
from baselines.baseline_memvr import BaselineMemVR
from utils.config import get_config

CONFIG_DIR = os.path.join(REPO_DIR, "configs")
config = get_config(CONFIG_DIR)

# B1/B3/B4 共用 GPU 0 的 base_model
b1 = BaselineDirect(base_model, base_processor)
b3 = BaselineMemVR(base_model, base_processor)
b4 = IVRInference(base_model, base_processor, config)

# B2 将在 GPU 1 独立加载 Thinking 模型
print("B1/B3/B4 initialized on GPU 0. B2 (Thinking) will load on GPU 1.")

## 5. 加载数据

在 Kaggle 右侧 Add Input > 搜索 `bdd100k` 或 `coco 2017` > 挂载后运行。

数据集来源：
- 主评测：BDD100K（自动驾驶场景，10万张，取500张）
- 补充评测：COCO val2017（通用视觉，500张）  
- 幻觉基准：POPE（独立评测集）

In [ ]:
QUESTION = "请详细描述这张图中的道路场景。是否有潜在危险（行人、障碍物、异常车辆）？如果有不确定的地方，请指出。"

## 6. 运行评测

In [ ]:
import time, json as json_module
from tqdm import tqdm
from pathlib import Path
from threading import Thread

CHECKPOINT_DIR = Path(REPO_DIR) / "results" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def load_checkpoint(name):
    ckpt_file = CHECKPOINT_DIR / f"{name}.json"
    if ckpt_file.exists():
        with open(ckpt_file) as f:
            data = json_module.load(f)
            return data.get("results", []), set(data.get("done_images", []))
    return [], set()

def save_checkpoint(name, results, done_images):
    ckpt_file = CHECKPOINT_DIR / f"{name}.json"
    tmp = CHECKPOINT_DIR / f"{name}.tmp"
    with open(tmp, "w") as f:
        json_module.dump({"results": results, "done_images": list(done_images)}, f, ensure_ascii=False)
    tmp.rename(ckpt_file)

results = {}
timing = {}

def run_baseline_gpu0(name, baseline, images, question):
    """在 GPU 0 上跑 B1/B3/B4"""
    prev, done = load_checkpoint(name)
    remaining = [(f, img) for f, img in images if f not in done]
    if not remaining:
        print(f"[{name}] 已完成，跳过")
        results[name] = prev
        timing[name] = {"status": "completed_earlier"}
        return
    print(f"\n[GPU 0] {name} | 已完成:{len(done)} 剩余:{len(remaining)}")
    r = prev.copy()
    start = time.time()
    for fname, img in tqdm(remaining, desc=name):
        try:
            result = baseline.generate(img, question)
            result["image"] = fname; result["baseline"] = name
            r.append(result); done.add(fname)
            if len(done) % 20 == 0:
                save_checkpoint(name, r, done)
        except Exception as e:
            tqdm.write(f"  Error: {e}")
    save_checkpoint(name, r, done)
    elapsed = time.time() - start
    avg_p = sum(x.get("num_passes",1) for x in r) / len(r) if r else 0
    results[name] = r
    timing[name] = {"total_seconds": round(elapsed,1), "avg_passes": round(avg_p,2)}

def run_baseline_gpu1(name, model, processor, images, question):
    """在 GPU 1 上跑 B2 Thinking 模型"""
    prev, done = load_checkpoint(name)
    remaining = [(f, img) for f, img in images if f not in done]
    if not remaining:
        print(f"[{name}] 已完成，跳过")
        results[name] = prev
        timing[name] = {"status": "completed_earlier"}
        return
    print(f"\n[GPU 1] {name} | 已完成:{len(done)} 剩余:{len(remaining)}")
    from baselines.baseline_cot import BaselineThinking
    baseline = BaselineThinking(model, processor)
    r = prev.copy()
    start = time.time()
    for fname, img in tqdm(remaining, desc=name):
        try:
            result = baseline.generate(img, question)
            result["image"] = fname; result["baseline"] = name
            r.append(result); done.add(fname)
            if len(done) % 20 == 0:
                save_checkpoint(name, r, done)
        except Exception as e:
            tqdm.write(f"  Error: {e}")
    save_checkpoint(name, r, done)
    elapsed = time.time() - start
    avg_p = sum(x.get("num_passes",1) for x in r) / len(r) if r else 0
    results[name] = r
    timing[name] = {"total_seconds": round(elapsed,1), "avg_passes": round(avg_p,2)}

# ── 加载 B2 到 GPU 1 ──
print("Loading Thinking model on GPU 1...")
thinking_model = AutoModelForImageTextToText.from_pretrained(
    THINKING_MODEL, trust_remote_code=True,
    torch_dtype=torch.float16, device_map="cuda:1",
)
thinking_processor = AutoProcessor.from_pretrained(THINKING_MODEL, trust_remote_code=True)
print(f"GPU 0: {torch.cuda.memory_allocated(0)/1e9:.1f}GB | GPU 1: {torch.cuda.memory_allocated(1)/1e9:.1f}GB")

# ── 双卡并行 ──
t0 = Thread(target=run_baseline_gpu1, args=("B2_Thinking", thinking_model, thinking_processor, images, QUESTION))
t0.start()

# GPU 0 串行 B1 → B3 → B4
for name, baseline in [("B1_Direct", b1), ("B3_MemVR", b3), ("B4_IVR", b4)]:
    run_baseline_gpu0(name, baseline, images, QUESTION)

t0.join()

print("\n" + "="*50)
print("All baselines complete! (2 GPU parallel)")
print(json_module.dumps(timing, indent=2, ensure_ascii=False))

## 7. 结果汇总

BDD100K 自动驾驶场景 + COCO 通用视觉 + POPE 幻觉探测

In [ ]:
import pandas as pd

summary_rows = []
for name, timing_info in timing.items():
    summary_rows.append({
        "Baseline": name,
        **timing_info,
        "num_images": len(results[name]) if name in results else 0,
    })

df = pd.DataFrame(summary_rows)
df = df.sort_values("avg_passes")
print("\n评测结果汇总:")
print(df.to_string(index=False))

## 8. 查看示例输出

对比各 baseline 在同一张图上的表现

In [ ]:
from IPython.display import display, Markdown

sample_idx = 0
baseline_names = ["B1_Direct", "B2_Thinking", "B3_MemVR", "B4_IVR"]
if all(name in results for name in ["B1_Direct", "B4_IVR"]):
    img_name = results["B1_Direct"][sample_idx]["image"]
    print(f"示例图像: {img_name}\n")
    
    for name in baseline_names:
        if name in results and sample_idx < len(results[name]):
            r = results[name][sample_idx]
            print(f"\n{'─'*40}")
            print(f"【{name}】(passes: {r.get('num_passes', 1)})")
            print(f"{'─'*40}")
            print(r.get("answer", "N/A")[:500])
            if "pass_confidences" in r:
                print(f"\n置信度: {r['pass_confidences']}")
            if "final_action" in r:
                print(f"终止原因: {r['final_action']}")

## 9. 生成对比图表

In [ ]:
from utils.visualization import plot_comparison_chart

# 注：CHAIR 和 POPE 分数需先运行对应评测脚本
# 此处用 timing 数据做示意

names = list(timing.keys())
avg_passes = [timing[n]["avg_passes"] for n in names]

plot_comparison_chart(
    baseline_names=names,
    chair_scores=[0.18, 0.12, 0.07, 0.04],  # 示例数据
    pope_scores=[0.71, 0.74, 0.79, 0.83],   # 示例数据
    avg_passes=avg_passes,
    output_path=f"{REPO_DIR}/results/comparison.png"
)

from IPython.display import Image as IPImage
IPImage(f"{REPO_DIR}/results/comparison.png")

## 10. 保存结果

In [ ]:
import json as json_module
from pathlib import Path

output_dir = Path(REPO_DIR) / "results"
output_dir.mkdir(parents=True, exist_ok=True)

output = {
    "config": {
        "model": MODEL_NAME,
        "num_images": len(images),
        "data_dir": DATA_DIR,
        "question": QUESTION,
    },
    "timing": timing,
}

with open(output_dir / "summary.json", "w") as f:
    json_module.dump(output, f, indent=2, ensure_ascii=False, default=str)

print(f"Results saved to {output_dir / 'summary.json'}")
print(f"Timing summary:")
total_time = sum(t["total_seconds"] for t in timing.values())
print(f"  Total runtime: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Baselines run: {len(timing)}")

---
## 附录：单独测试 IVR 的一次推理

用于快速验证单个样本

In [ ]:
# 快速测试单个样本
test_img = images[0][1] if images else None
if test_img:
    print(f"Testing on: {images[0][0]}")
    
    # B1: 直接回答
    r1 = b1.generate(test_img, QUESTION)
    print(f"\n[B1] Direct (1 pass):")
    print(f"  {r1['answer'][:200]}")
    
    # B4: IVR
    r4 = b4.generate(test_img, QUESTION)
    print(f"\n[B4] IVR ({r4['num_passes']} passes, conf={r4['final_confidence']:.2f}):")
    print(f"  {r4['answer'][:200]}")
    print(f"\n  Pass details:")
    for i, (ans, conf) in enumerate(zip(r4['pass_answers'], r4['pass_confidences'])):
        print(f"    Pass {i+1}: conf={conf:.3f} | {ans[:100]}")